# Final reproducible solution

Ноутбук для проверки организаторами. Он запускает решение от исходного файла `train_2.csv` (или `train.csv` того же формата) и формирует итоговый файл `test.csv` с колонками `new_id,rto`.

Важные свойства:

- фиксируется `RANDOM_SEED = 2026`;
- выводятся версии Python-библиотек и полный `pip freeze`;
- промежуточные таблицы и submission-кандидаты создаются во время выполнения notebook;
- notebook не использует `selected_solution/inputs/*.csv.gz` и другие заранее подготовленные meta-input CSV как вход решения;
- финальный выбор делается по конфигурации `huber030_g0.80_c0.008`, а не по номеру candidate ID, потому что при полном переобучении candidate ID может сдвигаться.


In [1]:
from pathlib import Path
import os
import random
import shutil
import sys
import platform
import subprocess
import importlib.metadata as metadata

import numpy as np
import pandas as pd

RANDOM_SEED = 2026
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().resolve()
while not (ROOT / "analysis" / "final_reproducible_candidate317.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

canonical_input = ROOT / "train_2.csv"
alternate_input = ROOT / "train.csv"
if not canonical_input.exists() and alternate_input.exists():
    shutil.copyfile(alternate_input, canonical_input)

assert canonical_input.exists(), "Expected train_2.csv or train.csv in project root"
assert (ROOT / "analysis" / "final_reproducible_candidate317.py").exists()

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Project root:", ROOT)
print("Input:", canonical_input)
print("RANDOM_SEED:", RANDOM_SEED)


Python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 10:07:17) [Clang 14.0.6 ]
Platform: macOS-15.5-arm64-arm-64bit
Project root: /Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient
Input: /Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient/train_2.csv
RANDOM_SEED: 2026


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Installed libraries

Сначала фиксируем версии ключевых библиотек, затем сохраняем полный `pip freeze` в `results/final_reproducible/package_versions.csv`.

In [2]:
key_packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "torch",
    "catboost",
    "joblib",
    "threadpoolctl",
    "nbformat",
    "nbclient",
    "notebook",
    "jupyter",
]

def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"

versions = pd.DataFrame({"package": key_packages, "version": [package_version(x) for x in key_packages]})
display(versions)

missing_required = versions[versions["package"].isin(["numpy", "pandas", "scikit-learn", "torch", "catboost"]) & versions["version"].eq("not installed")]
assert missing_required.empty, "Missing required packages: " + ", ".join(missing_required["package"].tolist())

freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines()
freeze_df = pd.DataFrame({"pip_freeze": freeze})
display(freeze_df)

out_dir = ROOT / "results" / "final_reproducible"
out_dir.mkdir(parents=True, exist_ok=True)
freeze_path = out_dir / "package_versions.csv"
freeze_df.to_csv(freeze_path, index=False)
print("Saved:", freeze_path.relative_to(ROOT))

,package,version
0,numpy,1.26.4
1,pandas,3.0.2
2,scikit-learn,1.4.2
3,scipy,1.13.1
4,torch,2.11.0
5,catboost,1.2.8
6,joblib,1.4.2
7,threadpoolctl,2.2.0
8,nbformat,5.9.2
9,nbclient,0.8.0


,pip_freeze
0,accelerate==1.13.0
1,ace_tools==0.0
2,addict==2.4.0
3,aext-assistant @ file:///private/var/folders/n...
4,aext-assistant-server @ file:///private/var/fo...
...,...
424,yarl==1.23.0
425,zict @ file:///Users/builder/cbouss/perseveran...
426,zipp @ file:///Users/builder/cbouss/perseveran...
427,zope.interface @ file:///Users/builder/cbouss/...


Saved: results/final_reproducible/package_versions.csv


## Run full pipeline

Запускаем полный pipeline. Он создает промежуточные файлы в `results/` и `submissions/`, но входом является только исходная история `train_2.csv` и код решения.

In [3]:
## Run full pipeline

Запускаем полный pipeline. Он создает промежуточные файлы в `results/` и `submissions/`, но входом является только исходная история `train_2.csv`/`train.csv` и код решения.



=== Running analysis/gru_residual_experiment.py ===


{
  "seq_len": 14,
  "epochs": 80,
  "best_gru": {
    "model": "gru",
    "alpha": 0.7,
    "blend_weight_gru": 1.0,
    "mape": 3.441718877580844
  },
  "comparison": [
    {
      "model": "gru",
      "alpha": 0.7,
      "blend_weight_gru": 1.0,
      "mape": 3.441718877580844
    },
    {
      "model": "gru",
      "alpha": 0.6,
      "blend_weight_gru": 1.0,
      "mape": 3.509379531349202
    },
    {
      "model": "gru",
      "alpha": 0.5,
      "blend_weight_gru": 1.0,
      "mape": 3.588823403590449
    },
    {
      "model": "gru",
      "alpha": 0.4,
      "blend_weight_gru": 1.0,
      "mape": 3.6784494059089234
    },
    {
      "model": "gru",
      "alpha": 0.3,
      "blend_weight_gru": 1.0,
      "mape": 3.7779479573016
    },
    {
      "model": "gru",
      "alpha": 0.2,
      "blend_weight_gru": 1.0,
      "mape": 3.886300788392839
    }
  ],
  "submission": "/Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient/submissions/test_gru_

Wrote calendar follow-up candidates:
 candidate_id                        name  val_mape_march2024  final_global_factor  test_mean_rto
           63        cal055_median_mom025            4.030440             1.129826   1.024648e+08
           64        cal060_median_mom025            4.064783             1.130855   1.025581e+08
           65       cal0625_median_mom025            4.084488             1.131420   1.026093e+08
           66        cal075_median_mom025            4.207942             1.133832   1.028281e+08
           67       cal0875_median_mom025            4.371154             1.136325   1.030541e+08
           68        cal100_median_mom025            4.570392             1.138705   1.032700e+08
           69       cal050_median_mom0125            3.907044             1.128838   1.029505e+08
           70        cal050_median_mom000            3.872334             1.128838   1.035295e+08
           71      cal0625_median_mom0125            3.946065             1.13142

OOF summary:
             variant  alpha       center     mape
        new_gru_none   1.00         none 3.330653
      new_gru_global   1.00       global 3.346709
        new_gru_none   0.85         none 3.357986
        new_gru_none   1.15         none 3.368107
      new_gru_global   0.85       global 3.370809
      new_gru_global   1.15       global 3.387441
new_gru_momentum_bin   1.00 momentum_bin 3.401702
        new_gru_none   0.70         none 3.418849
new_gru_momentum_bin   0.85 momentum_bin 3.428133
      new_gru_global   0.70       global 3.429938
new_gru_momentum_bin   1.15 momentum_bin 3.437200
        new_gru_none   1.30         none 3.455899
      new_gru_global   1.30       global 3.477895
new_gru_momentum_bin   0.70 momentum_bin 3.482570
new_gru_momentum_bin   1.30 momentum_bin 3.520906
        new_gru_none   0.50         none 3.522390
      new_gru_global   0.50       global 3.531127
new_gru_momentum_bin   0.50 momentum_bin 3.573013
        new_gru_none   0.30         n

Wrote high-gamma sweep candidates:
 candidate_id                                                         file                       name  calendar_lambda  gamma  clip  val_mape_march2024  test_mean_rto  test_total_rto  mean_abs_change_vs_base_pct  p95_abs_change_vs_base_pct
          114         stage2_candidate_114_baseline_cal050_mombin_g075.csv         cal050_mombin_g075             0.50   0.75                  3.555910   1.038035e+08    1.936662e+12                     1.037172                    2.270843
          115         stage2_candidate_115_baseline_cal050_mombin_g085.csv         cal050_mombin_g085             0.50   0.85                  3.522880   1.038469e+08    1.937471e+12                     1.187882                    2.576479
          116         stage2_candidate_116_baseline_cal050_mombin_g090.csv         cal050_mombin_g090             0.50   0.90                  3.507460   1.038692e+08    1.937888e+12                     1.264559                    2.730055
     

...

OOF summary:
         variant  gamma   clip     mape
second_cb_global   0.60  0.012 3.325559
second_cb_global   0.60   0.02 3.325567
second_cb_global   0.60        3.325567
second_cb_global   0.60  0.008 3.325601
   second_cb_raw   0.60  0.012 3.326239
   second_cb_raw   0.60        3.326242
   second_cb_raw   0.60   0.02 3.326242
second_cb_global   0.60  0.005 3.326251
   second_cb_raw   0.60  0.008 3.326280
   second_cb_raw   0.60  0.005 3.326993
second_cb_global   0.60  0.003 3.327632
second_cb_global   0.45  0.008 3.327832
second_cb_global   0.45  0.012 3.327836
second_cb_global   0.45   0.02 3.327836
second_cb_global   0.45        3.327836
second_cb_global   0.45  0.005 3.328038
   second_cb_raw   0.45  0.008 3.328205
   second_cb_raw   0.45   0.02 3.328206
   second_cb_raw   0.45  0.012 3.328206
   second_cb_raw   0.45        3.328206
   second_cb_raw   0.45  0.005 3.328449
   second_cb_raw   0.60  0.003 3.328474
second_cb_global   0.45  0.003 3.328897
   second_cb_raw   0.45  0.

OOF prediction diagnostics:
prediction  oof_mape  delta_vs_220
   pred204  3.836920      0.510771
   pred210  3.337326      0.011177
   pred118  3.478757      0.152608
   pred159  3.347351      0.021202
   pred215  3.327735      0.001586
   pred220  3.326149      0.000000

Submission candidates:
 candidate_id                                                                      file                              name  oof_mape  delta_vs_220_oof  mean_abs_vs_220_pct  p95_abs_vs_220_pct  max_abs_vs_220_pct  mean_delta_vs_220_pct  corr_with_250_corr
          261             stage2_candidate_261_full_v11_stack_huber030_g0p50_c0p005.csv             huber030_g0.50_c0.005  3.315752         -0.010397             0.179547            0.481745            0.501252              -0.004200                 NaN
          262             stage2_candidate_262_full_v11_stack_huber030_g0p75_c0p005.csv             huber030_g0.75_c0.005  3.315895         -0.010254             0.248224            0.501252     

Top v14 search:
                    name  oof_mape  delta_vs_265_oof  delta_vs_220_oof  train_share_changed  test_share_changed  mean_abs_vs_265_pct  p95_abs_vs_265_pct  max_abs_vs_265_pct  boot_p05  boot_p50  boot_p95  prob_improve
      top50_g1.00_c0.015  3.314520         -0.002352         -0.011629             0.500027            0.456611             0.044974            0.125690            1.005017 -0.003668 -0.002344 -0.001079         0.999
      top40_g1.00_c0.015  3.314575         -0.002297         -0.011574             0.400011            0.352147             0.039829            0.125690            1.005017 -0.003575 -0.002291 -0.001006         0.998
      top30_g1.00_c0.015  3.314662         -0.002211         -0.011487             0.299995            0.254489             0.033896            0.125690            1.005017 -0.003492 -0.002217 -0.000923         0.998
agree_top30_g1.00_c0.015  3.314696         -0.002177         -0.011454             0.253899            0.205982     

Wrote:
/Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient/results/temporal204_meta_inputs/oof_meta_inputs_v11_v14.csv
/Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient/results/temporal204_meta_inputs/test_meta_inputs_v11_v14.csv
/Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient/results/temporal204_meta_inputs/meta_input_oof_metrics.csv

=== Running analysis/new_baseline/train_v15_meta_residual_on_265.py ===


Features: 44 numeric, 5 categorical

Top search rows:
                 name   source  gamma  clip  oof_mape  delta_vs_265_oof  mean_abs_vs_265_pct  p95_abs_vs_265_pct  max_abs_vs_265_pct  mean_delta_vs_265_pct  boot_p05  boot_p50  boot_p95  prob_improve
huber030_g0.80_c0.012 huber030    0.8 0.012  3.283330         -0.033512             0.250820            0.713829            1.207229               0.018051 -0.038737 -0.033897 -0.028847           1.0
huber010_g0.80_c0.012 huber010    0.8 0.012  3.283517         -0.033325             0.252148            0.716175            1.207229               0.017193 -0.038555 -0.033707 -0.028464           1.0
huber030_g1.00_c0.012 huber030    1.0 0.012  3.283595         -0.033247             0.307516            0.892902            1.207229               0.019624 -0.039505 -0.033604 -0.027444           1.0
huber100_g0.80_c0.012 huber100    0.8 0.012  3.283659         -0.033183             0.254718            0.718949            1.207229              

## Validate output format

Проверяем, что итоговый файл `test.csv` подходит для загрузки на платформу.

In [4]:
submission_path = ROOT / "test.csv"
assert submission_path.exists(), "test.csv was not generated"
submission = pd.read_csv(submission_path)

print("Output:", submission_path)
print("Shape:", submission.shape)
print("Columns:", list(submission.columns))
print("NaN count:", int(submission.isna().sum().sum()))
print("Negative rto count:", int((submission["rto"] < 0).sum()))
display(submission.head())

assert submission.shape == (18657, 2)
assert list(submission.columns) == ["new_id", "rto"]
assert submission.isna().sum().sum() == 0
assert (submission["rto"] < 0).sum() == 0
print("OK: final test.csv is ready for submission")

Output: /Users/levonkrtyan/Library/Mobile Documents/com~apple~CloudDocs/Growth Gradient/test.csv
Shape: (18657, 2)
Columns: ['new_id', 'rto']
NaN count: 0
Negative rto count: 0


,new_id,rto
0,0,92066735.59
1,1,44565771.64
2,2,89667132.25
3,3,68875051.64
4,4,82817751.58


OK: final test.csv is ready for submission


## Notes

Для закрытой проверки организаторы могут заменить `train_2.csv` на файл того же формата с закрытой историей. Если файл будет называться `train.csv`, notebook автоматически создаст рабочий alias `train_2.csv`, потому что исторический код проекта ожидает именно это имя. Notebook заново построит признаки, обучит модели и сформирует новый `test.csv`.
